Injecting Parent & Child Chunks into SQL


In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("../.env"))

INPUT_FILE = Path("./chunks.json")

In [2]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

parent_chunks = [c for c in all_chunks if c["is_parent"]]
child_chunks = [c for c in all_chunks if not c["is_parent"]]

print(f"Loaded {len(all_chunks)} chunks from {INPUT_FILE}")
print(f"  - {len(parent_chunks)} parent chunks (for sparse search + LLM context)")
print(f"  - {len(child_chunks)} child chunks (metadata linking to Pinecone vectors)")

Loaded 825 chunks from chunks.json
  - 176 parent chunks (for sparse search + LLM context)
  - 649 child chunks (metadata linking to Pinecone vectors)


In [3]:
import asyncio
from config.settings import settings

print(f"Dev mode: {settings.dev_mode}")
print(f"Database: {settings.effective_database_url}")

Dev mode: True
Database: sqlite+aiosqlite:///./ragagent.db


In [4]:
from app.models.database import async_engine, async_session_factory, init_db

async def setup_tables():
    if settings.dev_mode:
        await init_db()
        print("SQLite tables created / verified")
    else:
        print("PostgreSQL mode — tables should already exist (via init_db.sql)")

await setup_tables()

2026-03-22 21:06:23,317 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-22 21:06:23,318 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("agent_sessions")
2026-03-22 21:06:23,318 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-22 21:06:23,320 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("agent_sessions")
2026-03-22 21:06:23,320 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-22 21:06:23,322 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("products")
2026-03-22 21:06:23,322 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-22 21:06:23,325 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("products")
2026-03-22 21:06:23,325 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-22 21:06:23,326 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("documents")
2026-03-22 21:06:23,327 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-22 21:06:23,328 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("documents")
2026-03-22 21:06:23,328 INFO sqlalchemy.engine

In [5]:
from sqlalchemy import text as sql_text

async def inject_chunks_to_sql(parents, children):
    inserted_parents = 0
    inserted_children = 0
    skipped = 0

    async with async_session_factory() as session:
        for pc in parents:
            try:
                await session.execute(
                    sql_text("""
                        INSERT INTO documents (id, title, source_url, doc_type,
                                               tenant_id, content, chunk_index,
                                               token_count, embedding_id)
                        VALUES (:id, :title, :source_url, :doc_type, :tenant_id,
                                :content, :chunk_index, :token_count, :embedding_id)
                    """),
                    {
                        "id": pc["chunk_id"],
                        "title": pc["filename"],
                        "source_url": None,
                        "doc_type": "document",
                        "tenant_id": None,
                        "content": pc["text"],
                        "chunk_index": pc["chunk_index"],
                        "token_count": pc["token_count"],
                        "embedding_id": None,
                    },
                )
                inserted_parents += 1
            except Exception as e:
                if "UNIQUE" in str(e).upper() or "duplicate" in str(e).lower():
                    skipped += 1
                else:
                    raise

        for cc in children:
            try:
                await session.execute(
                    sql_text("""
                        INSERT INTO documents (id, title, source_url, doc_type,
                                               tenant_id, content, parent_chunk_id,
                                               chunk_index, token_count,
                                               embedding_id)
                        VALUES (:id, :title, :source_url, :doc_type, :tenant_id,
                                :content, :parent_chunk_id, :chunk_index,
                                :token_count, :embedding_id)
                    """),
                    {
                        "id": cc["chunk_id"],
                        "title": cc["filename"],
                        "source_url": None,
                        "doc_type": "document",
                        "tenant_id": None,
                        "content": cc["text"],
                        "parent_chunk_id": cc["parent_id"],
                        "chunk_index": cc["chunk_index"],
                        "token_count": cc["token_count"],
                        "embedding_id": cc["chunk_id"],
                    },
                )
                inserted_children += 1
            except Exception as e:
                if "UNIQUE" in str(e).upper() or "duplicate" in str(e).lower():
                    skipped += 1
                else:
                    raise

        await session.commit()

    return inserted_parents, inserted_children, skipped

parents_ok, children_ok, skipped = await inject_chunks_to_sql(parent_chunks, child_chunks)

print(f"Inserted {parents_ok} parent chunks (sparse search + context)")
print(f"Inserted {children_ok} child chunks (metadata with parent_id link)")
if skipped:
    print(f"Skipped {skipped} duplicates")

2026-03-22 21:06:30,402 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-22 21:06:30,403 INFO sqlalchemy.engine.Engine 
                        INSERT INTO documents (id, title, source_url, doc_type,
                                               tenant_id, content, chunk_index,
                                               token_count, embedding_id)
                        VALUES (?, ?, ?, ?, ?,
                                ?, ?, ?, ?)
                    
2026-03-22 21:06:30,404 INFO sqlalchemy.engine.Engine [generated in 0.00059s] ('0503d079-7223-49eb-a3d7-b0c73a000457', '2507.19457v2.pdf', None, 'document', None, 'Accepted at ICLR 2026 (Oral).\nGEPA: REFLECTIVEPROMPTEVOLUTIONCANOUTPER-\nFORMREINFORCEMENTLEARNING\nLakshya A Agrawal1, Shangyin Tan1, Dilara Soylu2 ... (2507 characters truncated) ... \nfinal score. The Test-set star markers demonstrate the performance gap in a held-out set of questions.\n1\narXiv:2507.19457v2  [cs.CL]  14 Feb 2026', 0, 850, None)
2026-03-22 2

In [6]:
async def verify_injection():
    async with async_session_factory() as session:
        total = await session.execute(sql_text("SELECT COUNT(*) FROM documents"))
        total_count = total.scalar()

        parents = await session.execute(
            sql_text("SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NULL")
        )
        parent_count = parents.scalar()

        children = await session.execute(
            sql_text("SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NOT NULL")
        )
        child_count = children.scalar()

        sample = await session.execute(
            sql_text("SELECT id, title, token_count, parent_chunk_id FROM documents LIMIT 5")
        )
        rows = sample.fetchall()

    print(f"Documents table summary:")
    print(f"  Total rows:    {total_count}")
    print(f"  Parent chunks: {parent_count} (searchable via keyword/BM25)")
    print(f"  Child chunks:  {child_count} (linked to Pinecone vectors)")
    print(f"\nSample rows:")
    for row in rows:
        role = "child" if row[3] else "parent"
        print(f"  [{role:6}] {row[1]} — {row[2]} tokens")

await verify_injection()

2026-03-22 21:06:52,743 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-22 21:06:52,744 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM documents
2026-03-22 21:06:52,745 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-03-22 21:06:52,747 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NULL
2026-03-22 21:06:52,747 INFO sqlalchemy.engine.Engine [generated in 0.00060s] ()
2026-03-22 21:06:52,750 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM documents WHERE parent_chunk_id IS NOT NULL
2026-03-22 21:06:52,751 INFO sqlalchemy.engine.Engine [generated in 0.00069s] ()
2026-03-22 21:06:52,753 INFO sqlalchemy.engine.Engine SELECT id, title, token_count, parent_chunk_id FROM documents LIMIT 5
2026-03-22 21:06:52,753 INFO sqlalchemy.engine.Engine [generated in 0.00068s] ()
2026-03-22 21:06:52,755 INFO sqlalchemy.engine.Engine ROLLBACK
Documents table summary:
  Total rows:    825
  Parent chunks: 176 (searchable via keyword/BM25)
  